# CNN Studio — train SampleCNN

Upload this notebook into your empty Colab (`File → Upload notebook`).

**What you get:** `cnn_studio.pt` (~few hundred KB). Put it in your GitHub repo at `server/weights/cnn_studio.pt`. FastAPI loads it on boot.

**What the model does:** encode a 64×64 RGB crop → latent `z` → distribution over pose / light / warp. Views are **your pixels** resampled by a spatial transformer (identity-preserving). Not a face GAN.


In [ ]:
# GPU is nice but not required
!pip -q install torch pillow numpy matplotlib


In [ ]:
import math, io, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import display, Image as IPyImage
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

IMG_SIZE, LATENT_DIM, N_PARAMS = 64, 32, 10

def conv_block(cin, cout, stride=2):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.SiLU(inplace=True),
    )

class SampleCNN(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            conv_block(3, 24, 2), conv_block(24, 48, 2),
            conv_block(48, 72, 2), conv_block(72, 96, 2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.to_z = nn.Linear(96, latent_dim)
        self.head = nn.Sequential(nn.SiLU(), nn.Linear(latent_dim, 64), nn.SiLU(), nn.Linear(64, N_PARAMS * 2))
        last = self.head[-1]
        nn.init.zeros_(last.weight)
        mu = torch.zeros(N_PARAMS)
        log_std = torch.log(torch.tensor([0.12, 0.06, 0.06, 0.06, 0.05, 0.12, 0.12, 0.10, 0.08, 0.035]))
        with torch.no_grad():
            last.bias.copy_(torch.cat([mu, log_std]))

    def forward(self, x):
        z = self.to_z(self.encoder(x).flatten(1))
        raw = self.head(z)
        mu, log_std = raw.chunk(2, dim=-1)
        std = F.softplus(log_std) + 1e-3
        return z, mu, std

def affine_matrix(p):
    rot = p[:, 0].clamp(-0.45, 0.45)
    scale = (1.0 + p[:, 1]).clamp(0.82, 1.18)
    tx, ty = p[:, 2].clamp(-0.18, 0.18), p[:, 3].clamp(-0.18, 0.18)
    shear = p[:, 4].clamp(-0.18, 0.18)
    cos, sin = torch.cos(rot), torch.sin(rot)
    a = scale * (cos + shear * sin)
    b = scale * (-sin + shear * cos)
    row1 = torch.stack([a, b, tx], 1)
    row2 = torch.stack([scale * sin, scale * cos, ty], 1)
    return torch.stack([row1, row2], 1)

def color_jitter(x, p):
    b = p[:, 5].clamp(-0.35, 0.35)[:, None, None, None]
    c = (1.0 + p[:, 6].clamp(-0.35, 0.35))[:, None, None, None]
    s = (1.0 + p[:, 7].clamp(-0.35, 0.35))[:, None, None, None]
    h = p[:, 8].clamp(-0.25, 0.25)
    x = ((x - 0.5) * c + 0.5 + b).clamp(0, 1)
    gray = x.mean(1, keepdim=True)
    x = (gray + (x - gray) * s).clamp(0, 1)
    r, g, bl = x[:, 0], x[:, 1], x[:, 2]
    cr, sr = torch.cos(h), torch.sin(h)
    return torch.stack([cr * r + sr * g, -sr * r + cr * g, bl], 1).clamp(0, 1)

def elastic_grid(grid, amp):
    b, h, w, _ = grid.shape
    yy = torch.linspace(0, 3.1416, h, device=grid.device)
    xx = torch.linspace(0, 3.1416, w, device=grid.device)
    gy, gx = torch.meshgrid(yy, xx, indexing="ij")
    phase = amp.view(b, 1, 1)
    dx = 0.55 * phase * torch.sin(2 * gy) * torch.cos(gx)
    dy = 0.55 * phase * torch.cos(gy) * torch.sin(2 * gx)
    return grid + torch.stack([dx, dy], -1)

def apply_views(x, params):
    theta = affine_matrix(params)
    grid = F.affine_grid(theta, size=x.size(), align_corners=False)
    grid = elastic_grid(grid, params[:, 9].clamp(-0.12, 0.12))
    y = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    return color_jitter(y, params)

def sample_params(mu, std, n):
    if mu.dim() == 1:
        mu, std = mu.unsqueeze(0), std.unsqueeze(0)
    mu, std = mu.expand(n, -1), std.expand(n, -1)
    return mu + std * torch.randn(n, mu.size(-1), device=mu.device)

print("SampleCNN ready")


In [ ]:
# Optional: sklearn LFW portraits. If download fails, we train on synthetic ovals.
USE_LFW = True
faces = None
if USE_LFW:
    try:
        from sklearn.datasets import fetch_lfw_people
        lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)
        print("LFW", lfw.images.shape)
        faces = lfw.images  # HWC float
    except Exception as e:
        print("LFW skipped:", e)

def to_batch(arr, n=256):
    rng = np.random.default_rng(0)
    idx = rng.choice(len(arr), size=min(n, len(arr)), replace=False)
    out = []
    for i in idx:
        im = Image.fromarray((arr[i] * 255).astype(np.uint8) if arr[i].max() <= 1.5 else arr[i].astype(np.uint8))
        im = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        out.append(np.asarray(im, dtype=np.float32) / 255.0)
    t = torch.from_numpy(np.stack(out)).permute(0, 3, 1, 2)
    return t

def synthetic(n=256):
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
    cy, cx = IMG_SIZE / 2, IMG_SIZE / 2
    imgs = []
    for _ in range(n):
        skin, bg = rng.uniform(0.45, 0.9, 3), rng.uniform(0.05, 0.35, 3)
        img = np.tile(bg, (IMG_SIZE, IMG_SIZE, 1))
        face = ((xx - cx) / (IMG_SIZE * 0.28)) ** 2 + ((yy - cy) / (IMG_SIZE * 0.34)) ** 2 < 1
        img[face] = skin
        imgs.append(img)
    return torch.from_numpy(np.stack(imgs).astype(np.float32)).permute(0, 3, 1, 2)

data = to_batch(faces, 400) if faces is not None else synthetic(400)
print("train tensor", tuple(data.shape))


In [ ]:
model = SampleCNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS, BS = 8, 32
model.train()
for epoch in range(EPOCHS):
    perm = torch.randperm(data.size(0))
    losses = []
    for i in range(0, data.size(0) - BS + 1, BS):
        x = data[perm[i:i+BS]].to(device)
        z, mu, std = model(x)
        y = apply_views(x, sample_params(mu, std, x.size(0)))
        rec = F.l1_loss(y, x)
        loss = rec + 0.15 * mu.square().mean() + 0.05 * (std.mean() - 0.08).square() + 1e-4 * z.square().mean()
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f"epoch {epoch+1}/{EPOCHS}  loss={np.mean(losses):.4f}")
print("done, params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# Preview 16 views of a random training crop
model.eval()
with torch.no_grad():
    x = data[0:1].to(device)
    z, mu, std = model(x)
    params = sample_params(mu.squeeze(0), std.squeeze(0), 16)
    y = apply_views(x.expand(16, -1, -1, -1), params).cpu()
grid = y.permute(0, 2, 3, 1).numpy()
canvas = np.zeros((IMG_SIZE * 2, IMG_SIZE * 8, 3), dtype=np.float32)
for i, im in enumerate(grid):
    r, c = divmod(i, 8)
    canvas[r*IMG_SIZE:(r+1)*IMG_SIZE, c*IMG_SIZE:(c+1)*IMG_SIZE] = im
out = Image.fromarray((canvas.clip(0, 1) * 255).astype(np.uint8))
display(out)


In [ ]:
# Optional: upload YOUR photo and draw 64 CNN samples
print("Upload one portrait (skip this cell if you only want weights)")
up = files.upload()
if up:
    name = next(iter(up))
    im = Image.open(io.BytesIO(up[name])).convert("RGB")
    w, h = im.size
    s = min(w, h)
    im = im.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2)).resize((IMG_SIZE, IMG_SIZE))
    x = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        z, mu, std = model(x)
        n = 64
        y = apply_views(x.expand(n, -1, -1, -1), sample_params(mu.squeeze(0), std.squeeze(0), n)).cpu()
    cols = 8
    rows = n // cols
    canvas = np.zeros((rows * IMG_SIZE, cols * IMG_SIZE, 3), np.float32)
    for i, fr in enumerate(y.permute(0, 2, 3, 1).numpy()):
        r, c = divmod(i, cols)
        canvas[r*IMG_SIZE:(r+1)*IMG_SIZE, c*IMG_SIZE:(c+1)*IMG_SIZE] = fr
    display(Image.fromarray((canvas.clip(0,1)*255).astype(np.uint8)))


In [ ]:
path = "/content/cnn_studio.pt"
torch.save(model.cpu().state_dict(), path)
print("saved", path, "bytes", os.path.getsize(path))
files.download(path)
print("Next: put cnn_studio.pt in GitHub at server/weights/cnn_studio.pt then deploy FastAPI.")
